# Apache Hudi - Partitioning & Data Skipping

Partitioning and metadata indexing are critical for performance in large Hudi tables. Hudi can skip unnecessary files using metadata and column stats, improving query efficiency.

## What you will learn

In this notebook, you will learn:

- How partitioning works in Hudi
- Choosing a partition strategy
- Enabling Hudi metadata table
- Understanding data skipping
- Verifying pruning via EXPLAIN plan

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Hudi-08-Partitioning")
    .master("spark://spark-master:7077")
    .config("spark.serializer","org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions","org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .getOrCreate()
)

In [2]:
BASE_PATH="/workspace/data/hudi"
TABLE_NAME="riders_partitioning"
TABLE_PATH=f"{BASE_PATH}/{TABLE_NAME}"

In [3]:
import shutil,os
if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)

## Enable metadata & data skipping

In [4]:
spark.conf.set("spark.sql.hudi.metadata.enabled","true")
spark.conf.set("spark.sql.hudi.metadata.precise.skip","true")

## Write partitioned dataset

In [5]:
from pyspark.sql.functions import to_timestamp

df = spark.createDataFrame(
[
("r7","ATL","2024-01-09 10:00:00"),
("r8","MIA","2024-01-09 11:00:00"),
("r9","ATL","2024-01-09 12:00:00")
],["rider","city","ts"]).withColumn("ts",to_timestamp("ts"))

opts={
"hoodie.table.name":TABLE_NAME,
"hoodie.datasource.write.table.name":TABLE_NAME,
"hoodie.datasource.write.partitionpath.field":"city",
"hoodie.datasource.write.recordkey.field":"rider",
"hoodie.datasource.write.precombine.field":"ts",
"hoodie.datasource.write.table.type":"COPY_ON_WRITE"
}

df.write.format("hudi").options(**opts).mode("overwrite").save(TABLE_PATH)

# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope


[Stage 33:>                                                         (0 + 2) / 2]

26/04/25 19:52:38 WARN HoodieTableFileSystemView: Partition: ATL is not available in store
26/04/25 19:52:38 WARN HoodieTableFileSystemView: Partition: MIA is not available in store


## Query table

In [6]:
spark.read.format("hudi").load(TABLE_PATH).createOrReplaceTempView("riders_part")

spark.sql("SELECT * FROM riders_part").show()

+-------------------+--------------------+------------------+----------------------+--------------------+-----+----+-------------------+
|_hoodie_commit_time|_hoodie_commit_seqno|_hoodie_record_key|_hoodie_partition_path|   _hoodie_file_name|rider|city|                 ts|
+-------------------+--------------------+------------------+----------------------+--------------------+-----+----+-------------------+
|  20260425195217985|20260425195217985...|                r7|                   ATL|cfaac36b-0a16-44e...|   r7| ATL|2024-01-09 10:00:00|
|  20260425195217985|20260425195217985...|                r9|                   ATL|cfaac36b-0a16-44e...|   r9| ATL|2024-01-09 12:00:00|
|  20260425195217985|20260425195217985...|                r8|                   MIA|ebfb08d2-3e22-499...|   r8| MIA|2024-01-09 11:00:00|
+-------------------+--------------------+------------------+----------------------+--------------------+-----+----+-------------------+



## Verify partition pruning & data skipping

In [7]:
spark.sql("EXPLAIN SELECT rider FROM riders_part WHERE city='ATL'").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|plan                                                                                                                                                                                                                                                                                                                                                                                             |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------